# AML Stats — KPIs de negócio (últimos 7 dias)

Notebook batch que fecha o ciclo: lê as previsões geradas pelo consumer Kafka,
junta com a tabela de dimensão `dim_accounts` (derivada dos ficheiros `LI-*_accounts.csv`) e produz
indicadores de negócio agregados sobre uma **janela dos últimos 7 dias de ingest time**
(relativa a `current_timestamp()`).

Distinção:
- `Timestamp` (event time) — quando a transação ocorreu na realidade (fixo no dataset, 2022).
- `kafka_ts` / `ingest_ts` (ingest time) — quando o pipeline viu a mensagem (agora).

Também regista tabelas externas no metastore Hive (`aml_dim_accounts`, `aml_predictions`) para
consultas SQL ad-hoc.

**Pré-requisitos:**
1. `aml_trainer.ipynb` executado (modelo persistido).
2. `aml_kafka_consumer.ipynb` em execução (ou já tendo corrido) para produzir `predictions_kafka/`.
3. Ficheiros `LI-Small_accounts.csv` e `LI-Medium_accounts.csv` em HDFS na pasta `dataset/`.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
)

HDFS_BASE         = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project"
ACCOUNTS_PATH     = f"{HDFS_BASE}/dataset/LI-Medium_accounts.csv"
PRED_PATH         = f"{HDFS_BASE}/stream/predictions_kafka"
DIM_ACCOUNTS_PATH = f"{HDFS_BASE}/dim/accounts"
SPARK_MASTER = "spark://10.84.128.47:7077"

WINDOW_DAYS = 7

spark = SparkSession.builder \
    .master(SPARK_MASTER) \
    .appName("AML Stats") \
    .enableHiveSupport() \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

## 1) Tabela de dimensão `dim_accounts`

Construída a partir de `LI-Medium_accounts.csv` (o stream consome só o Medium, portanto Small
não traria nada de novo aos joins). `entity_type` é derivado por regex sobre `entity_name`
("Individual", "Corporation", "Partnership", "Sole Proprietorship", ...).

In [ ]:
accounts_schema = StructType([
    StructField("Bank Name",      StringType(),  True),
    StructField("Bank ID",        IntegerType(), True),
    StructField("Account Number", StringType(),  True),
    StructField("Entity ID",      StringType(),  True),
    StructField("Entity Name",    StringType(),  True),
])

raw_acc = spark.read \
    .option("header", True) \
    .schema(accounts_schema) \
    .csv(ACCOUNTS_PATH)

etype_extract = F.regexp_extract(F.col("Entity Name"), r"^(.+?)\s*#\d+", 1)

dim_accounts = (
    raw_acc
      .dropDuplicates(["Bank ID", "Account Number"])
      .withColumnRenamed("Bank ID",        "bank_id")
      .withColumnRenamed("Bank Name",      "bank_name")
      .withColumnRenamed("Account Number", "account_number")
      .withColumnRenamed("Entity ID",      "entity_id")
      .withColumnRenamed("Entity Name",    "entity_name")
      .withColumn(
          "entity_type",
          F.when(etype_extract == "", F.lit("Unknown")).otherwise(etype_extract),
      )
      .select("bank_id", "bank_name", "account_number", "entity_id", "entity_name", "entity_type")
)

dim_accounts.write.mode("overwrite").parquet(DIM_ACCOUNTS_PATH)
dim_accounts = spark.read.parquet(DIM_ACCOUNTS_PATH).cache()

print(f"dim_accounts: {dim_accounts.count():,} contas distintas")
dim_accounts.show(5, truncate=False)
print("Distribuição de entity_type:")
dim_accounts.groupBy("entity_type").count().orderBy(F.desc("count")).show(truncate=False)

## 2) Carregar previsões e filtrar aos últimos 7 dias

Lê as previsões geradas pelo `aml_kafka_consumer` e filtra a janela dos últimos 7 dias de
**ingest time** (`kafka_ts`) relativa a `current_timestamp()`. Todas as KPIs seguintes operam sobre
esta vista filtrada.

In [ ]:
preds_all = spark.read.parquet(PRED_PATH)

# Renomear kafka_ts -> ingest_ts para vocabulário uniforme
preds_all = preds_all.withColumnRenamed("kafka_ts", "ingest_ts")

# Janela: últimos N dias por ingest_ts
preds = preds_all.filter(
    F.col("ingest_ts") >= F.current_timestamp() - F.expr(f"INTERVAL {WINDOW_DAYS} DAYS")
)
preds.cache()

n_all = preds_all.count()
n_win = preds.count()
print(f"Previsões totais:                       {n_all:>10,}")
print(f"Janela: últimos {WINDOW_DAYS} dias por ingest_ts → {n_win:>10,} previsões")

## 3) KPIs globais — últimos 7 dias

In [ ]:
s = preds.agg(
    F.count("*").alias("total_tx"),
    F.sum("prediction").alias("n_suspeita"),
    F.sum(F.col("Amount Paid")).alias("valor_total"),
    F.sum(F.when(F.col("prediction") == 1, F.col("Amount Paid"))).alias("valor_suspeita"),
    F.sum("truth").alias("n_truth_fraude"),
    F.sum(F.when(F.col("truth") == 1, F.col("Amount Paid"))).alias("valor_truth_fraude"),
).collect()[0]

n_total   = s["total_tx"] or 0
n_susp    = s["n_suspeita"] or 0
n_ok      = n_total - n_susp
val_total = s["valor_total"] or 0
val_susp  = s["valor_suspeita"] or 0

print("=" * 60)
print(f"KPIs Globais — últimos {WINDOW_DAYS} dias")
print("=" * 60)
print(f"Total de transações analisadas:  {n_total:>14,}")
if n_total:
    print(f"  Fidedignas (prediction=0):     {n_ok:>14,}  ({n_ok/n_total:>6.2%})")
    print(f"  Suspeitas  (prediction=1):     {n_susp:>14,}  ({n_susp/n_total:>6.2%})")
print()
print(f"Valor total transacionado:       {val_total:>18,.2f}")
if val_total:
    print(f"Valor em transações suspeitas:   {val_susp:>18,.2f}  ({val_susp/val_total:>6.2%})")
print()
print(f"[ref. ground truth] fraude real: {s['n_truth_fraude'] or 0:>14,}  ({s['valor_truth_fraude'] or 0:>14,.2f})")

## 4) Fraude por instituição bancária

Join com `dim_accounts` por `(bank_id, account_number)` → traz nome real do banco.

In [ ]:
preds_enriched = (
    preds.alias("p")
         .join(
             dim_accounts.alias("a"),
             (F.col("p.From Bank") == F.col("a.bank_id")) &
             (F.col("p.From Account") == F.col("a.account_number")),
             how="left",
         )
         .select(
             "p.*",
             F.col("a.bank_name").alias("from_bank_name"),
             F.col("a.entity_id").alias("from_entity_id"),
             F.col("a.entity_name").alias("from_entity_name"),
             F.col("a.entity_type").alias("from_entity_type"),
         )
         .cache()
)

fraud_by_bank = (
    preds_enriched.filter("prediction = 1")
        .groupBy("from_bank_name")
        .agg(
            F.count("*").alias("n_tx_suspeitas"),
            F.sum(F.col("Amount Paid")).alias("valor_suspeito"),
            F.avg("prob_fraud").alias("prob_media"),
        )
        .orderBy(F.desc("valor_suspeito"))
)

print("Top 15 bancos por valor de transações suspeitas:")
fraud_by_bank.show(15, truncate=False)

## 5) Fraude por tipo de entidade

Individual (PF) vs Corporation / Partnership / Sole Proprietorship (PJ) — perfis de risco diferentes.

In [ ]:
fraud_by_entity_type = (
    preds_enriched.groupBy("from_entity_type")
        .agg(
            F.count("*").alias("n_tx"),
            F.sum("prediction").alias("n_suspeitas"),
            F.sum(F.when(F.col("prediction") == 1, F.col("Amount Paid"))).alias("valor_suspeito"),
        )
        .withColumn(
            "taxa_suspeita",
            F.when(F.col("n_tx") > 0, F.col("n_suspeitas") / F.col("n_tx")),
        )
        .orderBy(F.desc("valor_suspeito"))
)

print("Fraude por tipo de entidade:")
fraud_by_entity_type.show(truncate=False)

## 6) Top contas / entidades suspeitas

In [ ]:
top_accounts = (
    preds_enriched.filter("prediction = 1")
        .groupBy(
            F.col("From Account").alias("account"),
            "from_bank_name",
            "from_entity_name",
            "from_entity_type",
        )
        .agg(
            F.count("*").alias("n_suspeitas"),
            F.sum(F.col("Amount Paid")).alias("valor_suspeito"),
            F.avg("prob_fraud").alias("prob_media"),
        )
        .orderBy(F.desc("valor_suspeito"))
)

print("Top 20 contas com maior valor em transações suspeitas:")
top_accounts.show(20, truncate=False)

## 7) Tabelas externas Hive

Cria `aml_dim_accounts` e `aml_predictions` no metastore Hive (database `default`) para permitir
queries SQL ad-hoc. Cumpre o requisito do enunciado de "Spark, MapReduce, **Hive**, ...".

Se o cluster não tiver metastore Hive, o bloco falha sem afectar o resto do notebook.

In [ ]:
try:
    spark.sql("DROP TABLE IF EXISTS aml_dim_accounts")
    spark.sql(f"""
        CREATE EXTERNAL TABLE aml_dim_accounts (
          bank_id        INT,
          bank_name      STRING,
          account_number STRING,
          entity_id      STRING,
          entity_name    STRING,
          entity_type    STRING
        )
        STORED AS PARQUET
        LOCATION '{DIM_ACCOUNTS_PATH}'
    """)

    spark.sql("DROP TABLE IF EXISTS aml_predictions")
    spark.sql(f"""
        CREATE EXTERNAL TABLE aml_predictions (
          kafka_ts             TIMESTAMP,
          `Timestamp`          TIMESTAMP,
          `From Bank`          INT,
          `To Bank`            INT,
          `From Account`       STRING,
          `To Account`         STRING,
          `Amount Paid`        DOUBLE,
          `Receiving Currency` STRING,
          `Payment Format`     STRING,
          truth                INT,
          prediction           INT,
          prob_fraud           DOUBLE,
          status               STRING
        )
        PARTITIONED BY (ingest_hour STRING)
        STORED AS PARQUET
        LOCATION '{PRED_PATH}'
    """)
    spark.sql("MSCK REPAIR TABLE aml_predictions")

    print("Tabelas Hive criadas:")
    spark.sql("SHOW TABLES LIKE 'aml_*'").show()
except Exception as e:
    print(f"Hive metastore indisponível ou erro: {e}")
    print("As secções seguintes deste notebook ainda funcionam — só não terás SQL ad-hoc via Hive.")

## 8) Exemplo de query SQL via Hive

Demonstra que os dados já estão consultáveis com SQL puro pelo metastore.

In [ ]:
try:
    spark.sql(f"""
        WITH win AS (
            SELECT *
            FROM aml_predictions
            WHERE kafka_ts >= current_timestamp() - INTERVAL {WINDOW_DAYS} DAYS
        )
        SELECT a.entity_type,
               COUNT(*)                         AS n_suspeitas,
               ROUND(SUM(w.`Amount Paid`), 2)   AS valor_suspeito
        FROM win w
        JOIN aml_dim_accounts a
          ON w.`From Bank`    = a.bank_id
         AND w.`From Account` = a.account_number
        WHERE w.prediction = 1
        GROUP BY a.entity_type
        ORDER BY valor_suspeito DESC
    """).show(truncate=False)
except Exception as e:
    print(f"Query falhou (Hive indisponível?): {e}")